# 3. Data Cleaning


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import boxcox, zscore

from create_dataset import ds

import warnings
warnings.filterwarnings('ignore')

# Data cleaning

## What is Data
Data is information that's collected, organized, and analyzed to find patterns and meaning.

## How Data Comes?

- Tables

- Images

- Text

- Temporal

- Geospatial

Always remember
- it has to become a number eventually
- machines think in 0 or 1

## What is tabular data

- Continuous

- Categorical

- Ordinal

- Binary

- Time

## Iterate on data

- not only on models!
- data gathering, labelling, preparation are all iterative
- datasets are often fixed in academia, iterative in industry??


![Collecting_Data](./img/17_page.jpg)

## Characterizing data

Quality = outliers / missing values

Quantity = rows & columns

Diversity = does the distribution match the test set

Cardinality - number of unique values

Duplicates

Class imbalance

Label noise

Biased sampling

![Collecting_Data_3](./img/19_page.jpg)

## Sample dataset

A simple dataset:

In [4]:
ds

,customers-category,contract-length,location
0,A,10.0,us
1,A,9.0,us
2,A,9.0,us
3,B,20.0,nz
4,B,30.0,NaN
5,B,10.0,nz
6,C,NaN,nz
7,C,5.0,nz
8,C,NaN,us
9,NaN,1.0,nz


## Duplicates

Important to not have duplicates in both your test & train data

- but you could leave duplicate samples if they are only in the train data: "true" duplicates
- the repetitions provide weight of evidence (especially important in Bayesian methods)
- the frequency infomation is important

Calling `.duplicated()` returns a boolean mask:

In [5]:
ds.duplicated()

0     False
1     False
2      True
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
dtype: bool

We can use `.duplicated()` to filter out the duplicate:

In [6]:
ds[~ds.duplicated()]

,customers-category,contract-length,location
0,A,10.0,us
1,A,9.0,us
3,B,20.0,nz
4,B,30.0,NaN
5,B,10.0,nz
6,C,NaN,nz
7,C,5.0,nz
8,C,NaN,us
9,NaN,1.0,nz
10,C,100.0,nz


## Missing values

Not always a `np.Nan` - sometimes this can be encoded as a `0` or a `null` string
- know how missing is represented in your dataset

Always ask - **why is this data missing?**
- is there bias (a pattern) in the missing values?

Dealing with missing values = **detection + replacement**

Detection depends on how the missing value is encoded:

In [7]:
for col in ds.columns:
    print(col, sum(ds.loc[:, col].isnull()))

customers-category 1
contract-length 2
location 1


### Dropping rows or columns

One solution to missing data is to drop the row (sample) or the column (feature)

- quick & dirty way to deal with missing values
- lose data
- should be done on the first iteration (if working in an agile way)

In [8]:
ds.dropna(axis=0) #axis=0 is row, axis=1 is column

,customers-category,contract-length,location
0,A,10.0,us
1,A,9.0,us
2,A,9.0,us
3,B,20.0,nz
5,B,10.0,nz
7,C,5.0,nz
10,C,100.0,nz


### Imputation for continuous variables

Using a statistic like mean or median:

In [9]:
data = ds.loc[:, 'contract-length']  #previously we used the simple-imputer
filled = data.fillna(round(data.mean()))

pd.concat([data, filled], axis=1)

,contract-length,contract-length
0,10.0,10.0
1,9.0,9.0
2,9.0,9.0
3,20.0,20.0
4,30.0,30.0
5,10.0,10.0
6,NaN,22.0
7,5.0,5.0
8,NaN,22.0
9,1.0,1.0


An improvement may be to fill with a statistic that is conditional on another feature:

In [10]:
ds

,customers-category,contract-length,location
0,A,10.0,us
1,A,9.0,us
2,A,9.0,us
3,B,20.0,nz
4,B,30.0,NaN
5,B,10.0,nz
6,C,NaN,nz
7,C,5.0,nz
8,C,NaN,us
9,NaN,1.0,nz


In [11]:
data = ds.loc[:, ['contract-length', 'customers-category']]
new = pd.DataFrame()

for customer in set(data.loc[:, 'customers-category'].dropna()):
    #  get rows with this customer type
    mask = data.loc[:, 'customers-category'] == customer
    
    #  fill the na's using the mean of this customer type
    filled = data[mask].copy()
    filled.loc[:, 'contract-length'].fillna(np.mean(data.loc[mask, 'contract-length']), inplace=True)
    new = pd.concat([new, filled] ).sort_values(by=['customers-category'])

pd.concat([data, new], axis=1)

#I would prefer sklearn.SimpleImputer though

,contract-length,customers-category,contract-length,customers-category
0,10.0,A,10.0,A
1,9.0,A,9.0,A
2,9.0,A,9.0,A
3,20.0,B,20.0,B
4,30.0,B,30.0,B
5,10.0,B,10.0,B
6,NaN,C,52.5,C
7,5.0,C,5.0,C
8,NaN,C,52.5,C
9,1.0,NaN,NaN,NaN


Another solution is to **train a model to predict the missing value**
- perhaps a linear model using other columns as features

### Imputation for categorical variables

Most common class (similar to mean / median)

Missing value token
- tell the model it was missing

Fill with the most common label conditional on another feature (as with continuous)

Train a model to predict it (as with continuous)

# Outliers

Within distribution outliers, versus outliers that are due to mistakes / measurement errors

Require both detection & replacement
- detection is harder than missing values

### Detecting outliers

Using standard deviation
- multiple on the z-score (2 - 4)

Though Z-Score is a highly efficient way of detecting and removing outliers, we cannot use it with every data type. When we say that, we mean that it only works with the data which is completely or close to normally distributed, which in turn stimulates that this method is not for skewed data, either left skew or right skew


![z_score](./images/440971.png)

In [12]:
zsc = zscore(ds.loc[:, 'contract-length'].fillna(np.mean(ds.loc[:, 'contract-length'])))
zsc_mask = zsc > 3.0

pd.concat([
    ds, 
    pd.DataFrame({'zsc_mask': zsc_mask, 'zscore': zsc})
], axis=1)

,customers-category,contract-length,location,zsc_mask,zscore
0,A,10.0,us,False,-0.442361
1,A,9.0,us,False,-0.480642
2,A,9.0,us,False,-0.480642
3,B,20.0,nz,False,-0.059549
4,B,30.0,NaN,False,0.323264
5,B,10.0,nz,False,-0.442361
6,C,NaN,nz,False,0.000000
7,C,5.0,nz,False,-0.633767
8,C,NaN,us,False,0.000000
9,NaN,1.0,nz,False,-0.786891


Detection using percentiles
- always lose data 

In [13]:
#  calculate the 95th percentile
pct = np.percentile(ds.loc[:, 'contract-length'].fillna(np.mean(ds.loc[:, 'contract-length'])), 95)
pct

np.float64(65.0)

In [14]:
#  drop all rows less than this number
ds.loc[ds.loc[:, 'contract-length'] < pct]

,customers-category,contract-length,location
0,A,10.0,us
1,A,9.0,us
2,A,9.0,us
3,B,20.0,nz
4,B,30.0,NaN
5,B,10.0,nz
7,C,5.0,nz
9,NaN,1.0,nz


Clipping / capping
- this will change the distribution

In [15]:
max_val = 50
np.clip(ds.loc[:, 'contract-length'], None, max_val)

0     10.0
1      9.0
2      9.0
3     20.0
4     30.0
5     10.0
6      NaN
7      5.0
8      NaN
9      1.0
10    50.0
Name: contract-length, dtype: float64

## String cleaning

Consistent naming
- U.K., UK, United Kingdom, Britian, Great Britian etc

Tokenization
- breaking text into smaller chunks (often splitting on spaces)
- word or sentence level
- Test on https://tiktokenizer.vercel.app

Normalization
- removing stop words
- lower case
- stemming / lemming

## LLM-Assisted Data Cleaning

In 2026, a common first step in cleaning is to ask an LLM to draft a diagnosis or a first-pass script — then **critically review its output** rather than trust it blindly. The same "anatomy of a good prompt" applies here as anywhere else:

| Component | Purpose | Example for cleaning |
|---|---|---|
| **System prompt** | Set the model's role | "You are a data cleaning assistant." |
| **Task instruction** | Exactly what to do | "List likely data quality issues in this dataframe sample." |
| **Context** | Relevant background | The `.head()` and `.dtypes` output of your dataframe |
| **Output format** | Shape of the answer | "Return a bullet list, one issue per line." |

**The critical-review step is the actual skill** — treat the LLM's suggestions as a first draft, then verify each one against the data yourself.

In [16]:
# ── LLM-assisted cleaning: simulated for teaching, no API key required ──
# (see 06_LLMs_and_Foundation_Models.ipynb for the real API call pattern)

def simulate_llm_cleaning_suggestions(df_sample):
    """Simulates asking an LLM to spot likely data quality issues."""
    return [
        "`customers-category` has a missing value in one row — decide: drop, impute, or flag?",
        "`contract-length` has missing values and at least one extreme outlier (100) vs the rest (1-30) — worth a sanity check with a domain expert.",
        "`location` has a missing value — check whether missingness correlates with `customers-category`.",
    ]

suggestions = simulate_llm_cleaning_suggestions(ds.head())
print("LLM-suggested issues to verify:")
for s in suggestions:
    print(f"  - {s}")

print("\n👉 Now go verify each suggestion against the real data yourself -- don't take it on faith.")

LLM-suggested issues to verify:
  - `customers-category` has a missing value in one row — decide: drop, impute, or flag?
  - `contract-length` has missing values and at least one extreme outlier (100) vs the rest (1-30) — worth a sanity check with a domain expert.
  - `location` has a missing value — check whether missingness correlates with `customers-category`.

👉 Now go verify each suggestion against the real data yourself -- don't take it on faith.


## Scaling

Scaling (min-max, standardization, robust scaling, Box-Cox, etc.) matters a lot for many models — but the full treatment, with worked examples, lives in **ML Fundamentals → `Scaling and Outliers.ipynb`**. For today, just remember the golden rule:

> Fit your scaler on the training set only, then apply the same fitted scaler to validation/test data. Never fit on data your model hasn't earned the right to see yet.


## Fixing target labeling

Most targets have errors/noise.

Sometimes these can be fixed - it depends on the business problem